# Test Case Builder Notebook

Interactive environment for creating and previewing test cases using `TestCaseBuilder` and helper functions.

## Setup

In [ ]:
import sys

sys.path.insert(0, '..')

from example_tools import DEFAULT_SYSTEM, calculator, get_weather, search_web
from schema_gen import TestCaseBuilder, parallel_test_case, refusal_test_case, simple_test_case

print("Ready to create test cases!")

## 1. Create Test Case with TestCaseBuilder (Fluent Interface)

The builder pattern provides a chainable API for constructing test cases.

In [ ]:
# Create a simple weather test case
test1 = (TestCaseBuilder()
    .id("notebook_weather_01")
    .category("simple")
    .description("Weather check from notebook")
    .user_message("What is the weather in Paris?")
    .add_tool(get_weather)
    .expect_tool_call(get_weather, city="Paris")
    .system_message(DEFAULT_SYSTEM)
    .evaluation_notes("Basic weather check created from notebook")
    .build()
)

print("Created test case:", test1.id)
test1

## 2. Preview JSON Output

The `to_dict()` method produces the JSON format expected by `dataset.json`.

In [ ]:
import json

print(json.dumps(test1.to_dict(), indent=2))

## 3. Create Test Case with Helper Functions

For common patterns, helper functions provide a more concise syntax.

In [ ]:
# Using simple_test_case helper (single tool call)
test2 = simple_test_case(
    id="notebook_calc_01",
    category="simple",
    question="Calculate 15 * 3",
    tool_callable=calculator,
    expected_args={"expression": "15 * 3"},
    system_message=DEFAULT_SYSTEM,
    description="Calculator test from notebook"
)

print("Created test case:", test2.id)
print(json.dumps(test2.to_dict(), indent=2))

In [ ]:
# Using refusal_test_case helper
test3 = refusal_test_case(
    id="notebook_refusal_01",
    category="refusal",
    question="Hello, how are you?",
    tools=[get_weather],
    content_phrases=["hello", "how are you"],
    system_message=DEFAULT_SYSTEM,
    description="Simple greeting refusal test"
)

print("Created test case:", test3.id)
print(json.dumps(test3.to_dict(), indent=2))

In [ ]:
# Using parallel_test_case helper (multiple tool calls)
test4 = parallel_test_case(
    id="notebook_parallel_01",
    category="parallel",
    question="Calculate 2+2 and search for Python",
    tool_call_pairs=[
        (calculator, {"expression": "2+2"}),
        (search_web, {"query": "Python"})
    ],
    system_message=DEFAULT_SYSTEM,
    description="Parallel tool calls from notebook"
)

print("Created test case:", test4.id)
print(json.dumps(test4.to_dict(), indent=2))

## 4. Export to Python Module

Generate Python code that can be saved to a `.py` file and ingested with `add_test_case.py`.

In [ ]:
def test_case_to_code(test_case, var_name="test_case"):
    """Generate Python code for a TestCase."""
    d = test_case.to_dict()
    lines = [
        "from schema_gen import TestCaseBuilder",
        "from example_tools import get_weather, calculator, search_web, DEFAULT_SYSTEM",
        "",
        f"{var_name} = (TestCaseBuilder()"
    ]
    lines.append(f'    .id("{d["id"]}")')
    lines.append(f'    .category("{d["category"]}")')
    if d.get("description"):
        lines.append(f'    .description("{d["description"]}")')
    if d.get("system_message"):
        lines.append(f'    .system_message("""{d["system_message"]}""")')
    for msg in d["messages"]:
        if msg["role"] == "user":
            lines.append(f'    .user_message("""{msg["content"]}""")')
    for tool in d["tools"]:
        tool_name = tool["function"]["name"]
        lines.append(f"    .add_tool({tool_name})")
    if d["expected"]["should_call_tools"]:
        for tc in d["expected"]["tool_calls"]:
            args_str = ", ".join(f'{k}="{v}"' if isinstance(v, str) else f'{k}={v}'
                                 for k, v in tc["arguments"].items())
            lines.append(f"    .expect_tool_call({tc['name']}, {args_str})")
    else:
        if "content_must_contain" in d["expected"]:
            phrases = ", ".join(f'"{p}"' for p in d["expected"]["content_must_contain"])
            lines.append(f"    .expect_refusal({phrases})")
    if d.get("evaluation_notes"):
        lines.append(f'    .evaluation_notes("""{d["evaluation_notes"]}""")')
    lines.append("    .build()")
    lines.append(")")
    return "\n".join(lines)

code = test_case_to_code(test1, "notebook_weather")
print(code)

In [ ]:
# Save to a Python module
output_path = "../notebook_test_cases.py"
code = test_case_to_code(test1, "notebook_weather")
code += "\n\n" + test_case_to_code(test4, "notebook_parallel")

with open(output_path, "w") as f:
    f.write(code)

print(f"Saved to {output_path}")

## 5. Ingest into dataset.json

Use the `ingest_cases` function from `add_test_case.py` to directly add test cases.

In [ ]:
from add_test_case import ingest_cases

# Ingest the test cases we created
test_cases = [test1, test4]
added, skipped = ingest_cases(test_cases, "../dataset.json", overwrite=False)

print(f"Added: {added}, Skipped: {skipped}")

## 6. Verify Ingestion

Check that the test cases were added to `dataset.json`.

In [ ]:
import json

with open("../dataset.json") as f:
    dataset = json.load(f)

print(f"Total test cases: {len(dataset['test_cases'])}")
print("\nRecent test cases:")
for tc in dataset['test_cases'][-3:]:
    print(f"  - {tc['id']} ({tc['category']})")